# LAB 3 — Bài 2: Phân loại chủ đề văn bản bằng GRU trên UIT-VSFC

> **Phiên bản hướng dẫn dành cho sinh viên mới học**  
> Notebook trình bày đầy đủ quy trình xây dựng mô hình phân loại văn bản: đọc dữ liệu, tiền xử lý, tạo vocabulary, tạo batch có padding, xây dựng GRU, huấn luyện, chọn checkpoint tốt nhất và đánh giá trên test set.

## Mục tiêu học tập

Sau khi hoàn thành notebook, sinh viên có thể:

1. Hiểu cách biểu diễn câu bằng chuỗi chỉ số token.
2. Phân biệt token đặc biệt `<pad>` và `<unk>`.
3. Xây dựng `Vocabulary`, `Dataset`, `collate_fn` và `DataLoader`.
4. Hiểu vai trò của `Embedding`, GRU và hidden state.
5. Phân biệt GRU với RNN cơ bản và LSTM ở mức kiến trúc.
6. Xử lý chuỗi có độ dài khác nhau bằng padding và `pack_padded_sequence`.
7. Viết vòng lặp huấn luyện và đánh giá đúng chuẩn PyTorch.
8. Theo dõi loss, accuracy, precision, recall và macro F1-score.
9. Lưu, tải lại checkpoint tốt nhất và phân tích các mẫu dự đoán sai.

---

## Quy trình chuẩn của bài toán

```text
Tệp JSON
   │
   ├── Train set ──> xây vocabulary + cập nhật trọng số
   │
   ├── Dev set ──> chọn epoch và siêu tham số tốt nhất
   │
   └── Test set ──> chỉ dùng để báo cáo kết quả cuối cùng
```

**Lưu ý quan trọng:** Vocabulary chỉ được xây từ **train set**. Nếu dùng cả dev hoặc test để xây vocabulary, mô hình đã gián tiếp nhìn thấy thông tin từ dữ liệu đánh giá.


## 1. Cài đặt thư viện

Notebook sử dụng:

- `torch`: xây dựng và huấn luyện mạng GRU.
- `scikit-learn`: tính các chỉ số đánh giá và confusion matrix.
- `matplotlib`: trực quan hóa dữ liệu và quá trình học.
- Các thư viện chuẩn như `json`, `re`, `pathlib` để đọc và xử lý dữ liệu.

Trong Jupyter Notebook, nên dùng `%pip` để cài thư viện vào đúng môi trường Python của notebook.

In [ ]:
# Chỉ cần chạy ô này nếu môi trường chưa có các thư viện cần thiết.
%pip install -q scikit-learn matplotlib

## 2. Import, thiết lập seed và chọn thiết bị

### Random seed là gì?

Khởi tạo trọng số, xáo trộn dữ liệu và Dropout đều có yếu tố ngẫu nhiên. Đặt seed giúp các lần chạy cho kết quả gần giống nhau, thuận tiện khi học tập và so sánh thí nghiệm.

### CPU hay GPU?

PyTorch sẽ dùng GPU nếu CUDA khả dụng. Mô hình và mọi tensor tham gia phép tính phải nằm trên cùng một thiết bị.

In [ ]:
from collections import Counter
from functools import partial
from pathlib import Path
import json
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int = 42) -> None:
    """Thiết lập seed để tăng khả năng tái lập kết quả."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Ưu tiên tính tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Thiết bị đang dùng: {device}")

## 3. Khai báo và kiểm tra đường dẫn dữ liệu

Notebook giả sử thư mục có cấu trúc:

```text
LAB3.ipynb
UIT-VSFC/
├── UIT-VSFC-train.json
├── UIT-VSFC-dev.json
└── UIT-VSFC-test.json
```

Kiểm tra tệp ngay từ đầu giúp báo lỗi rõ ràng, thay vì để chương trình lỗi ở bước tạo dataset hoặc huấn luyện.

In [ ]:
DATA_DIR = Path("UIT-VSFC")
TRAIN_PATH = DATA_DIR / "UIT-VSFC-train.json"
DEV_PATH = DATA_DIR / "UIT-VSFC-dev.json"
TEST_PATH = DATA_DIR / "UIT-VSFC-test.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [TRAIN_PATH, DEV_PATH, TEST_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    missing_text = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Không tìm thấy các tệp UIT-VSFC sau:\n"
        f"{missing_text}\n\n"
        "Hãy đặt thư mục 'UIT-VSFC' cùng cấp với notebook "
        "hoặc sửa biến DATA_DIR."
    )

print("Đã tìm thấy đầy đủ các tệp train, dev và test.")

## 4. Đọc và kiểm tra dữ liệu JSON

Mỗi phần tử dữ liệu được kỳ vọng có ít nhất hai trường:

- `sentence`: nội dung câu nhận xét.
- `topic`: nhãn chủ đề cần dự đoán.

Không nên giả định dữ liệu luôn đúng. Việc kiểm tra kiểu dữ liệu và các trường bắt buộc giúp phát hiện lỗi sớm.

In [ ]:
def load_json_records(path: Path) -> list[dict]:
    """Đọc một tệp JSON và kiểm tra cấu trúc cơ bản."""
    with path.open("r", encoding="utf-8") as file:
        records = json.load(file)

    if not isinstance(records, list):
        raise ValueError(f"Dữ liệu trong '{path}' phải là một danh sách.")

    for index, record in enumerate(records):
        if not isinstance(record, dict):
            raise ValueError(
                f"Phần tử {index} trong '{path}' không phải dictionary."
            )

        missing_keys = {"sentence", "topic"} - set(record)
        if missing_keys:
            raise ValueError(
                f"Phần tử {index} trong '{path}' thiếu trường: "
                f"{sorted(missing_keys)}"
            )

        if not isinstance(record["sentence"], str):
            raise TypeError(
                f"Trường 'sentence' tại phần tử {index} phải là chuỗi."
            )

    return records


train_records = load_json_records(TRAIN_PATH)
dev_records = load_json_records(DEV_PATH)
test_records = load_json_records(TEST_PATH)

print(f"Train set: {len(train_records):,} mẫu")
print(f"Dev set:   {len(dev_records):,} mẫu")
print(f"Test set:  {len(test_records):,} mẫu")
print("\nVí dụ một mẫu train:")
print(train_records[0])

## 5. Tiền xử lý văn bản và xây dựng vocabulary

### Tiền xử lý trong notebook này

Mỗi câu được:

1. Chuyển về chữ thường.
2. Thay dấu câu bằng khoảng trắng.
3. Xóa khoảng trắng thừa.
4. Tách token bằng khoảng trắng.

Đây là cách tiền xử lý đơn giản để minh họa quy trình. Với tiếng Việt, sinh viên có thể thử thêm công cụ tách từ chuyên biệt ở phần bài tập mở rộng.

### Hai token đặc biệt

- `<pad>`: bổ sung vào cuối câu ngắn để các câu trong cùng batch có cùng chiều dài.
- `<unk>`: thay cho token không xuất hiện trong vocabulary của train set.

### Vì sao không xây vocabulary từ dev và test?

Dev và test là dữ liệu đánh giá. Dùng chúng để quyết định vocabulary gây **rò rỉ dữ liệu**. Vì vậy, vocabulary và ánh xạ nhãn chỉ được học từ train set.

In [ ]:
def preprocess_text(text: str) -> str:
    """Chuẩn hóa văn bản bằng các thao tác cơ bản."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    """Tiền xử lý và tách câu thành danh sách token."""
    normalized_text = preprocess_text(text)
    return normalized_text.split() if normalized_text else []


class Vocabulary:
    """Ánh xạ token thành chỉ số, chỉ xây dựng từ train set."""

    PAD_TOKEN = "<pad>"
    UNK_TOKEN = "<unk>"

    def __init__(
        self,
        sentences: list[str],
        min_frequency: int = 1,
    ) -> None:
        token_counts = Counter()

        for sentence in sentences:
            token_counts.update(tokenize(sentence))

        # Sắp xếp để vocabulary ổn định giữa các lần chạy:
        # token có tần suất cao đứng trước, sau đó theo alphabet.
        vocabulary_tokens = sorted(
            (
                token
                for token, frequency in token_counts.items()
                if frequency >= min_frequency
            ),
            key=lambda token: (-token_counts[token], token),
        )

        self.word_to_index = {
            self.PAD_TOKEN: 0,
            self.UNK_TOKEN: 1,
        }

        for token in vocabulary_tokens:
            self.word_to_index[token] = len(self.word_to_index)

        self.index_to_word = {
            index: token
            for token, index in self.word_to_index.items()
        }

        self.pad_index = self.word_to_index[self.PAD_TOKEN]
        self.unk_index = self.word_to_index[self.UNK_TOKEN]

    def __len__(self) -> int:
        return len(self.word_to_index)

    def encode(self, sentence: str) -> list[int]:
        """Chuyển một câu thành danh sách chỉ số token."""
        token_ids = [
            self.word_to_index.get(token, self.unk_index)
            for token in tokenize(sentence)
        ]

        # Tránh chuỗi có độ dài bằng 0 sau tiền xử lý.
        return token_ids if token_ids else [self.unk_index]

    def decode(self, token_ids: list[int]) -> list[str]:
        """Chuyển danh sách chỉ số trở lại token để kiểm tra."""
        return [
            self.index_to_word.get(token_id, self.UNK_TOKEN)
            for token_id in token_ids
        ]


train_sentences = [record["sentence"] for record in train_records]
vocab = Vocabulary(train_sentences, min_frequency=1)

train_labels = sorted({record["topic"] for record in train_records})
label_to_index = {
    label: index
    for index, label in enumerate(train_labels)
}
index_to_label = {
    index: label
    for label, index in label_to_index.items()
}

unknown_dev_labels = {
    record["topic"] for record in dev_records
} - set(label_to_index)
unknown_test_labels = {
    record["topic"] for record in test_records
} - set(label_to_index)

if unknown_dev_labels or unknown_test_labels:
    raise ValueError(
        "Dev hoặc test chứa nhãn không xuất hiện trong train set. "
        f"Dev: {sorted(unknown_dev_labels)} | "
        f"Test: {sorted(unknown_test_labels)}"
    )

print(f"Kích thước vocabulary: {len(vocab):,}")
print(f"Số lớp: {len(label_to_index)}")
print(f"Ánh xạ nhãn: {label_to_index}")

## 6. Kiểm tra kết quả mã hóa một câu

Trước khi tạo mô hình, nên xem trực tiếp:

- Câu gốc.
- Các token sau tiền xử lý.
- Danh sách chỉ số.
- Kết quả giải mã ngược.

Bước này giúp phát hiện lỗi tiền xử lý hoặc ánh xạ vocabulary.

In [ ]:
sample_sentence = train_records[0]["sentence"]
sample_token_ids = vocab.encode(sample_sentence)

print(f"Câu gốc:       {sample_sentence}")
print(f"Token:         {tokenize(sample_sentence)}")
print(f"Token IDs:     {sample_token_ids}")
print(f"Giải mã lại:   {vocab.decode(sample_token_ids)}")
print(f"Nhãn chủ đề:   {train_records[0]['topic']}")
print(f"Chỉ số nhãn:   {label_to_index[train_records[0]['topic']]}")

## 7. Xây dựng `Dataset`

Một lớp kế thừa `Dataset` cần tối thiểu:

- `__len__`: trả về số lượng mẫu.
- `__getitem__`: trả về một mẫu tại vị trí `index`.

Mỗi mẫu trong notebook gồm:

- `input_ids`: tensor chứa chỉ số token.
- `label_id`: chỉ số nhãn.
- `sentence`: câu gốc để phân tích lỗi sau này.

Nhãn phải có kiểu `torch.long` vì `CrossEntropyLoss` nhận chỉ số lớp nguyên.

In [ ]:
class VSFCDataset(Dataset):
    """Dataset cho bài toán phân loại chủ đề UIT-VSFC."""

    def __init__(
        self,
        records: list[dict],
        vocabulary: Vocabulary,
        label_mapping: dict,
    ) -> None:
        self.records = records
        self.vocabulary = vocabulary
        self.label_mapping = label_mapping

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        record = self.records[index]

        input_ids = torch.tensor(
            self.vocabulary.encode(record["sentence"]),
            dtype=torch.long,
        )
        label_id = torch.tensor(
            self.label_mapping[record["topic"]],
            dtype=torch.long,
        )

        return {
            "input_ids": input_ids,
            "label_id": label_id,
            "sentence": record["sentence"],
        }


train_dataset = VSFCDataset(train_records, vocab, label_to_index)
dev_dataset = VSFCDataset(dev_records, vocab, label_to_index)
test_dataset = VSFCDataset(test_records, vocab, label_to_index)

sample = train_dataset[0]
print(f"Shape input_ids của một mẫu: {sample['input_ids'].shape}")
print(f"Kiểu input_ids: {sample['input_ids'].dtype}")
print(f"Label ID: {sample['label_id'].item()}")

## 8. Tạo batch có padding bằng `collate_fn`

Các câu có số token khác nhau nên không thể ghép trực tiếp thành một tensor hình chữ nhật.

Ví dụ:

```text
Câu 1: [12, 25, 9, 41]
Câu 2: [ 7, 18]
```

Sau padding:

```text
Câu 1: [12, 25,  9, 41]
Câu 2: [ 7, 18, <pad>, <pad>]
```

`collate_fn` thực hiện ba việc:

1. Ghi lại độ dài thật của từng câu.
2. Padding các câu đến độ dài lớn nhất trong batch.
3. Ghép nhãn thành một tensor.

Độ dài thật sẽ được dùng với `pack_padded_sequence`, giúp GRU bỏ qua phần `<pad>`.

In [ ]:
def collate_batch(
    items: list[dict],
    pad_index: int,
) -> dict:
    """Ghép các câu có độ dài khác nhau thành một batch."""
    sequences = [item["input_ids"] for item in items]
    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        dtype=torch.long,
    )

    padded_input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=pad_index,
    )

    label_ids = torch.stack([item["label_id"] for item in items])
    sentences = [item["sentence"] for item in items]

    return {
        "input_ids": padded_input_ids,
        "lengths": lengths,
        "label_ids": label_ids,
        "sentences": sentences,
    }

## 9. Tạo `DataLoader`

`DataLoader`:

1. Gom nhiều mẫu thành một batch.
2. Xáo trộn train set.
3. Gọi `collate_fn` để padding câu.

Chỉ train loader dùng `shuffle=True`. Dev và test không cập nhật trọng số nên không cần xáo trộn.

In [ ]:
BATCH_SIZE = 32
PIN_MEMORY = device.type == "cuda"

batch_collator = partial(
    collate_batch,
    pad_index=vocab.pad_index,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=batch_collator,
    pin_memory=PIN_MEMORY,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=batch_collator,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=batch_collator,
    pin_memory=PIN_MEMORY,
)

sample_batch = next(iter(train_loader))

print(f"Shape input_ids: {sample_batch['input_ids'].shape}")
print(f"Shape lengths:   {sample_batch['lengths'].shape}")
print(f"Shape label_ids: {sample_batch['label_ids'].shape}")
print(f"Số batch train mỗi epoch: {len(train_loader)}")
print(f"Độ dài thật của 5 câu đầu: {sample_batch['lengths'][:5].tolist()}")

## 10. Khảo sát phân bố nhãn

Macro F1 cho mỗi lớp vai trò ngang nhau, nên đặc biệt hữu ích khi các lớp không cân bằng.

Trước khi huấn luyện, cần xem:

- Mỗi nhãn có bao nhiêu mẫu.
- Nhãn nào chiếm ưu thế.
- Nhãn nào có ít dữ liệu.

In [ ]:
train_label_counts = Counter(
    record["topic"] for record in train_records
)

print("Số mẫu train theo nhãn:")
for label in train_labels:
    print(f"- {label}: {train_label_counts[label]:,}")

plt.figure(figsize=(10, 5))
plt.bar(
    [str(label) for label in train_labels],
    [train_label_counts[label] for label in train_labels],
)
plt.xlabel("Nhãn chủ đề")
plt.ylabel("Số mẫu")
plt.title("Phân bố nhãn trong train set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Lý thuyết mô hình GRU

### 11.1. Embedding

Mỗi token ban đầu chỉ là một số nguyên. Lớp `Embedding` biến mỗi chỉ số thành một vector có thể học:

```text
token ID → vector embedding
```

Nếu batch có shape `[B, T]`, sau embedding ta được `[B, T, E]`:

- `B`: batch size.
- `T`: độ dài chuỗi sau padding.
- `E`: embedding dimension.

### 11.2. GRU

GRU là một biến thể của mạng nơ-ron hồi tiếp, sử dụng hai cổng chính:

- **Update gate**: quyết định giữ lại bao nhiêu thông tin từ hidden state trước.
- **Reset gate**: quyết định bỏ qua bao nhiêu thông tin cũ khi tạo hidden state mới.

Khác với LSTM, GRU không duy trì một cell state riêng. Vì có ít cổng hơn, GRU thường có ít tham số và có thể huấn luyện nhanh hơn LSTM, nhưng mô hình nào tốt hơn vẫn phụ thuộc dữ liệu và cách thiết lập thí nghiệm.

### 11.3. Bidirectional GRU

- GRU xuôi đọc từ đầu đến cuối.
- GRU ngược đọc từ cuối về đầu.
- Hai hidden state cuối của layer cuối được nối lại trước khi phân loại.

### 11.4. Vì sao dùng `pack_padded_sequence`?

Nếu đưa trực tiếp chuỗi đã padding vào GRU, hidden state có thể bị ảnh hưởng bởi các bước `<pad>`. Packed sequence cho GRU biết độ dài thật để bỏ qua phần padding.

### 11.5. Đầu ra của GRU trong PyTorch

`nn.GRU` trả về hai thành phần:

```text
output, hidden_states = gru(input)
```

GRU không trả về cell state như LSTM. Với mô hình hai chiều, hai phần tử cuối của `hidden_states` là hidden state xuôi và ngược của layer cuối cùng.


In [ ]:
class GRUClassifier(nn.Module):
    """Mô hình phân loại văn bản bằng Embedding và GRU."""

    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        pad_index: int,
        embedding_dim: int = 128,
        hidden_dim: int = 128,
        num_layers: int = 2,
        bidirectional: bool = True,
        dropout_rate: float = 0.30,
    ) -> None:
        super().__init__()

        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_index,
        )
        self.embedding_dropout = nn.Dropout(dropout_rate)

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_rate if num_layers > 1 else 0.0,
        )

        output_dim = hidden_dim * (2 if bidirectional else 1)
        self.output_dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(output_dim, num_classes)

    def forward(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor,
    ) -> torch.Tensor:
        embeddings = self.embedding(input_ids)
        embeddings = self.embedding_dropout(embeddings)

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        # GRU trả về output và hidden state; không có cell state.
        _, hidden_states = self.gru(packed_embeddings)

        if self.bidirectional:
            # Hai phần tử cuối tương ứng với chiều xuôi và ngược
            # của layer GRU cuối cùng.
            forward_hidden = hidden_states[-2]
            backward_hidden = hidden_states[-1]
            features = torch.cat(
                [forward_hidden, backward_hidden],
                dim=1,
            )
        else:
            features = hidden_states[-1]

        features = self.output_dropout(features)
        logits = self.classifier(features)
        return logits


def count_trainable_parameters(model: nn.Module) -> int:
    """Đếm số tham số được cập nhật khi huấn luyện."""
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


## 12. Chọn mô hình và siêu tham số

Các siêu tham số quan trọng:

- `EMBEDDING_DIM`: số chiều vector biểu diễn token.
- `HIDDEN_DIM`: số chiều hidden state của GRU.
- `NUM_LAYERS`: số tầng GRU xếp chồng.
- `BIDIRECTIONAL`: có đọc câu theo cả hai chiều hay không.
- `DROPOUT_RATE`: tỷ lệ Dropout để giảm overfitting.
- `EPOCHS`: số lần đi qua toàn bộ train set.
- `LEARNING_RATE`: độ lớn bước cập nhật.
- `PATIENCE`: số epoch chờ dev F1 cải thiện trước khi dừng sớm.

Mô hình quá lớn có thể học chậm hoặc overfit. Nên bắt đầu với cấu hình vừa phải rồi mới thử tăng kích thước.

In [ ]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT_RATE = 0.30

EPOCHS = 20
LEARNING_RATE = 1e-3
PATIENCE = 4
MAX_GRAD_NORM = 5.0

model_config = {
    "vocab_size": len(vocab),
    "num_classes": len(label_to_index),
    "pad_index": vocab.pad_index,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "bidirectional": BIDIRECTIONAL,
    "dropout_rate": DROPOUT_RATE,
}

model = GRUClassifier(**model_config).to(device)

print(model)
print(f"Số tham số có thể học: {count_trainable_parameters(model):,}")

## 13. Loss function, optimizer và gradient clipping

### `CrossEntropyLoss`

Mô hình trả về **logits**, không thêm Softmax trước loss. `CrossEntropyLoss` đã kết hợp LogSoftmax và Negative Log-Likelihood theo cách ổn định số học.

### Optimizer Adam

Adam thường là lựa chọn dễ bắt đầu cho bài toán NLP cơ bản.

### Gradient clipping

RNN và GRU có thể gặp hiện tượng gradient quá lớn. `clip_grad_norm_` giới hạn chuẩn gradient trước khi optimizer cập nhật trọng số.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

CHECKPOINT_PATH = OUTPUT_DIR / "best_gru_vsfc.pth"
print(f"Checkpoint sẽ được lưu tại: {CHECKPOINT_PATH.resolve()}")

## 14. Hàm huấn luyện một epoch

Quy trình của mỗi batch:

1. Chuyển tensor lên thiết bị.
2. Xóa gradient cũ.
3. Forward để tính logits.
4. Tính loss.
5. Backward để tính gradient.
6. Clip gradient.
7. Cập nhật trọng số.

Loss trung bình được tính theo số mẫu để batch cuối nhỏ hơn không làm sai trung bình.

In [ ]:
def train_one_epoch(
    dataloader: DataLoader,
    model: nn.Module,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_grad_norm: float,
) -> float:
    """Huấn luyện đúng một epoch và trả về loss trung bình."""
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True,
        )
        lengths = batch["lengths"]
        label_ids = batch["label_ids"].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(input_ids, lengths)
        loss = loss_fn(logits, label_ids)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_grad_norm,
        )
        optimizer.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

## 15. Hàm đánh giá

Khi đánh giá:

- `model.eval()` tắt hành vi ngẫu nhiên của Dropout.
- `torch.inference_mode()` không lưu đồ thị gradient, giúp tiết kiệm bộ nhớ.
- `argmax(dim=1)` chọn lớp có logit lớn nhất.

Notebook theo dõi:

- Accuracy.
- Macro precision.
- Macro recall.
- Macro F1-score.

Macro F1 được dùng để chọn checkpoint vì mỗi lớp được tính trọng số ngang nhau.

In [ ]:
def evaluate(
    dataloader: DataLoader,
    model: nn.Module,
    loss_fn: nn.Module,
    device: torch.device,
) -> dict:
    """Đánh giá mô hình và trả về loss, metrics cùng dự đoán."""
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_predictions = []
    all_sentences = []

    with torch.inference_mode():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True,
            )
            lengths = batch["lengths"]
            label_ids = batch["label_ids"].to(
                device,
                non_blocking=True,
            )

            logits = model(input_ids, lengths)
            loss = loss_fn(logits, label_ids)
            predictions = logits.argmax(dim=1)

            batch_size = input_ids.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_labels.extend(label_ids.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_sentences.extend(batch["sentences"])

    all_labels = np.asarray(all_labels)
    all_predictions = np.asarray(all_predictions)

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(all_labels, all_predictions),
        "precision": precision_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0,
        ),
        "recall": recall_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0,
        ),
        "f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0,
        ),
        "labels": all_labels,
        "predictions": all_predictions,
        "sentences": all_sentences,
    }

## 16. Huấn luyện và lưu checkpoint tốt nhất

Sau mỗi epoch:

1. Huấn luyện trên train set.
2. Đánh giá trên dev set.
3. Nếu dev F1 tốt hơn, lưu checkpoint.
4. Nếu dev F1 không cải thiện trong nhiều epoch, dừng sớm.

Checkpoint lưu cả cấu hình, trọng số, optimizer và ánh xạ nhãn. Điều này giúp việc tải lại mô hình rõ ràng và ít nhầm lẫn hơn.

In [ ]:
history = {
    "train_loss": [],
    "dev_loss": [],
    "dev_accuracy": [],
    "dev_f1": [],
}

best_dev_f1 = -np.inf
epochs_without_improvement = 0
training_start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        train_loader,
        model,
        loss_fn,
        optimizer,
        device,
        MAX_GRAD_NORM,
    )

    dev_result = evaluate(
        dev_loader,
        model,
        loss_fn,
        device,
    )

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_result["loss"])
    history["dev_accuracy"].append(dev_result["accuracy"])
    history["dev_f1"].append(dev_result["f1"])

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Dev loss: {dev_result['loss']:.4f} | "
        f"Dev accuracy: {dev_result['accuracy'] * 100:.2f}% | "
        f"Dev F1: {dev_result['f1'] * 100:.2f}%"
    )

    if dev_result["f1"] > best_dev_f1:
        best_dev_f1 = dev_result["f1"]
        epochs_without_improvement = 0

        torch.save(
            {
                "model_config": model_config,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_dev_f1": best_dev_f1,
                "epoch": epoch,
                "word_to_index": vocab.word_to_index,
                "label_to_index": label_to_index,
            },
            CHECKPOINT_PATH,
        )
        print("  ↳ Đã lưu checkpoint tốt nhất.")
    else:
        epochs_without_improvement += 1
        print(
            "  ↳ Dev F1 chưa cải thiện: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Dừng sớm vì dev F1 không cải thiện "
            f"trong {PATIENCE} epoch liên tiếp."
        )
        break

training_time = time.time() - training_start_time
print(f"\nThời gian huấn luyện: {training_time:.1f} giây")
print(f"Dev F1 tốt nhất: {best_dev_f1 * 100:.2f}%")

## 17. Vẽ biểu đồ quá trình học

Một quá trình học hợp lý thường có:

- Train loss giảm dần.
- Dev loss giảm hoặc ổn định.
- Dev F1 tăng rồi bão hòa.

Nếu train loss tiếp tục giảm nhưng dev loss tăng và dev F1 giảm, mô hình có thể đang overfit.

In [ ]:
epoch_numbers = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(
    epoch_numbers,
    history["train_loss"],
    marker="o",
    label="Train loss",
)
plt.plot(
    epoch_numbers,
    history["dev_loss"],
    marker="o",
    label="Dev loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss theo epoch")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    epoch_numbers,
    np.array(history["dev_accuracy"]) * 100,
    marker="o",
    label="Dev accuracy",
)
plt.plot(
    epoch_numbers,
    np.array(history["dev_f1"]) * 100,
    marker="o",
    label="Dev macro F1",
)
plt.xlabel("Epoch")
plt.ylabel("Tỷ lệ (%)")
plt.title("Chỉ số trên dev set theo epoch")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 18. Tải lại checkpoint tốt nhất

Mô hình trong biến `model` sau vòng lặp là mô hình ở epoch cuối cùng, chưa chắc có dev F1 tốt nhất.

Trước khi đánh giá test set, phải tải lại checkpoint được chọn bằng dev set. `map_location=device` giúp checkpoint hoạt động cả khi huấn luyện bằng GPU nhưng tải trên CPU.

In [ ]:
try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=True,
    )
except TypeError:
    # Tương thích với phiên bản PyTorch cũ chưa có weights_only.
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
    )

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Đã tải checkpoint tại epoch: {checkpoint['epoch']}")
print(
    "Dev F1 của checkpoint: "
    f"{checkpoint['best_dev_f1'] * 100:.2f}%"
)

## 19. Đánh giá cuối cùng trên test set

Test set chỉ được dùng sau khi đã hoàn tất việc chọn mô hình bằng dev set.

Nếu tiếp tục thay đổi kiến trúc hoặc siêu tham số dựa trên test score, test set sẽ dần mất tính khách quan và trở thành một dev set thứ hai.

In [ ]:
test_result = evaluate(
    test_loader,
    model,
    loss_fn,
    device,
)

ordered_label_ids = np.arange(len(index_to_label))
ordered_label_names = [
    str(index_to_label[index])
    for index in ordered_label_ids
]

print("KẾT QUẢ TRÊN TEST SET")
print("-" * 45)
print(f"Test loss:       {test_result['loss']:.4f}")
print(f"Test accuracy:   {test_result['accuracy'] * 100:.2f}%")
print(f"Macro precision: {test_result['precision'] * 100:.2f}%")
print(f"Macro recall:    {test_result['recall'] * 100:.2f}%")
print(f"Macro F1-score:  {test_result['f1'] * 100:.2f}%")

print("\nBáo cáo theo từng lớp:\n")
print(
    classification_report(
        test_result["labels"],
        test_result["predictions"],
        labels=ordered_label_ids,
        target_names=ordered_label_names,
        digits=4,
        zero_division=0,
    )
)

## 20. Confusion matrix

Confusion matrix cho biết:

- Hàng: nhãn thật.
- Cột: nhãn dự đoán.
- Đường chéo chính: số dự đoán đúng.
- Ô ngoài đường chéo: các cặp lớp thường bị nhầm.

Đây là công cụ quan trọng để tìm lớp khó và định hướng cải thiện dữ liệu hoặc mô hình.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    test_result["labels"],
    test_result["predictions"],
    labels=ordered_label_ids,
    display_labels=ordered_label_names,
    cmap="Blues",
    values_format="d",
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion matrix trên test set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 21. Xem một số câu dự đoán sai

Chỉ số tổng hợp không giải thích được nguyên nhân. Đọc trực tiếp các câu sai giúp đặt câu hỏi:

- Câu có mơ hồ giữa nhiều chủ đề không?
- Câu có quá ngắn hoặc thiếu ngữ cảnh không?
- Có nhiều từ ngoài vocabulary không?
- Hai lớp có nội dung gần nhau không?

In [ ]:
def show_misclassified_sentences(
    result: dict,
    index_to_label: dict,
    number_of_samples: int = 10,
) -> None:
    """In một số câu bị mô hình dự đoán sai."""
    wrong_indices = np.where(
        result["labels"] != result["predictions"]
    )[0]

    if len(wrong_indices) == 0:
        print("Không có mẫu dự đoán sai.")
        return

    number_of_samples = min(
        number_of_samples,
        len(wrong_indices),
    )

    for order, sample_index in enumerate(
        wrong_indices[:number_of_samples],
        start=1,
    ):
        true_id = int(result["labels"][sample_index])
        predicted_id = int(result["predictions"][sample_index])

        print(f"[{order}] {result['sentences'][sample_index]}")
        print(f"    Nhãn thật: {index_to_label[true_id]}")
        print(f"    Dự đoán:   {index_to_label[predicted_id]}")
        print("-" * 80)


show_misclassified_sentences(
    test_result,
    index_to_label,
    number_of_samples=10,
)

## 22. Dự đoán một câu mới

Khi dự đoán một câu:

1. Dùng đúng hàm tiền xử lý và vocabulary đã xây từ train set.
2. Thêm chiều batch bằng `unsqueeze(0)`.
3. Dùng Softmax chỉ để chuyển logits thành xác suất dễ đọc.
4. Không dùng Softmax trước `CrossEntropyLoss` khi huấn luyện.

In [ ]:
def predict_topic(
    model: nn.Module,
    sentence: str,
    vocabulary: Vocabulary,
    index_to_label: dict,
    device: torch.device,
) -> tuple[object, float]:
    """Dự đoán nhãn chủ đề và độ tin cậy của một câu."""
    token_ids = torch.tensor(
        vocabulary.encode(sentence),
        dtype=torch.long,
    )
    input_ids = token_ids.unsqueeze(0).to(device)
    lengths = torch.tensor([len(token_ids)], dtype=torch.long)

    model.eval()
    with torch.inference_mode():
        logits = model(input_ids, lengths)
        probabilities = torch.softmax(logits, dim=1)
        confidence, predicted_id = probabilities.max(dim=1)

    predicted_label = index_to_label[predicted_id.item()]
    return predicted_label, confidence.item()


new_sentence = "Giảng viên giảng bài rất dễ hiểu và nhiệt tình."
predicted_topic, confidence = predict_topic(
    model,
    new_sentence,
    vocab,
    index_to_label,
    device,
)

print(f"Câu: {new_sentence}")
print(f"Chủ đề dự đoán: {predicted_topic}")
print(f"Độ tin cậy: {confidence * 100:.2f}%")

## 23. Tổng kết kiến thức

### Quy trình đúng

1. Kiểm tra cấu trúc dữ liệu.
2. Chỉ xây vocabulary và ánh xạ nhãn từ train set.
3. Dùng `<unk>` cho token chưa từng thấy.
4. Tạo `Dataset`, `collate_fn` và padding theo batch.
5. Truyền độ dài thật vào `pack_padded_sequence`.
6. Xây mô hình GRU trả về logits.
7. Huấn luyện với `model.train()`.
8. Đánh giá với `model.eval()` và `torch.inference_mode()`.
9. Chọn checkpoint bằng dev macro F1.
10. Tải checkpoint tốt nhất rồi mới đánh giá test set.

### Những lỗi người mới thường gặp

- Xây vocabulary từ cả train, dev và test.
- Dùng cùng một chỉ số cho `<unk>` và token bắt đầu câu.
- Không xử lý câu rỗng sau tiền xử lý.
- Để GRU đọc cả phần padding rồi lấy hidden state cuối.
- Nhầm cách giải nén đầu ra của GRU với đầu ra của LSTM.
- Xử lý sai hidden state khi dùng bidirectional GRU.
- Thêm Softmax trước `CrossEntropyLoss`.
- Quên xóa gradient trước mỗi batch.
- Quên gradient clipping khi mạng hồi tiếp học không ổn định.
- Dùng test set để chọn epoch.
- Chỉ lưu trọng số epoch cuối thay vì checkpoint tốt nhất.

---

## Bài tập mở rộng

1. So sánh GRU một chiều và bidirectional GRU.
2. Thử `embedding_dim` và `hidden_dim`: `64`, `128`, `256`.
3. Thử `num_layers`: `1`, `2`, `3` và nhận xét thời gian huấn luyện.
4. Thử các learning rate: `1e-2`, `1e-3`, `1e-4`.
5. Chỉ giữ token xuất hiện ít nhất 2 hoặc 3 lần bằng `min_frequency`.
6. Dùng class weights trong `CrossEntropyLoss` nếu nhãn mất cân bằng.
7. Thử tokenizer tiếng Việt và so sánh macro F1.
8. Tìm ba cặp nhãn bị nhầm nhiều nhất từ confusion matrix.
9. Thử LSTM với cùng siêu tham số, sau đó so sánh số tham số, tốc độ và macro F1.
10. Thử pretrained word embedding hoặc một mô hình Transformer.

### Câu hỏi tự kiểm tra

- Vì sao vocabulary chỉ được xây từ train set?
- `<pad>` và `<unk>` khác nhau như thế nào?
- Vì sao các câu trong một batch cần padding?
- `pack_padded_sequence` giải quyết vấn đề gì?
- Update gate và reset gate của GRU có vai trò gì?
- GRU khác LSTM ở đầu ra và trạng thái bộ nhớ như thế nào?
- Hidden state của bidirectional GRU được ghép như thế nào?
- Vì sao đầu ra mô hình không cần Softmax khi huấn luyện?
- Vì sao macro F1 phù hợp khi các lớp không cân bằng?
- Dev set và test set có vai trò khác nhau như thế nào?
